# 机票价格及推荐工具

## 练习目标（理念）

用 **第 1 天** 学到的 Chat Completions 思路，做一个有趣的小推荐器：先拉（或模拟）航班报价，再让模型按偏好给出 Top 3 推荐。

**航班数据** 来自 [Amadeus Open API](https://developers.amadeus.com/)（测试环境 / 免费套餐）。没有密钥时会自动走 **mock** 假数据，仍可练通 LLM 调用。

该工具根据以下因素推荐机票价格和航空公司：

- **用户偏好**（最实惠、物有所值或最佳航空公司）
- **出发地与目的地**（IATA 机场码，如 `LAS` → `LHR`）
- **旅行起止日期 / 时长**
- **成人与儿童人数**

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量 + `.env` | `OPENROUTER_API_KEY`、Amadeus 凭证 |
| HTTP API | `requests` 调 Amadeus OAuth + flight-offers |
| Chat Completions | OpenRouter 上的 `openai/gpt-4o-mini` |
| Markdown 展示 | `display(Markdown(result))` |

## 怎么跑

1. 在 `.env` 里配置 `OPENROUTER_API_KEY`；可选配置 `AMADEUS_CLIENT_ID` / `AMADEUS_CLIENT_SECRET`
2. 从上到下运行；无 Amadeus 凭证时打印 `mock`，仍可用假航班数据做推荐
3. 改最后一格的城市码、日期、人数与 `preference` 再跑


## 第 1 周 · 第 1 天挑战

下面从导入与 Amadeus HTTP，到 `TicketRecommender` 类，再到一次完整演示。


In [ ]:
# ========== 导入 + Amadeus 端点常量 ==========

# 导入标准库 os：读环境变量（API Key、Amadeus 凭证）
import os
# 导入标准库 json：把结构化数据 dumps 进发给模型的 user prompt
import json
# 从 datetime 导入 datetime：用日期字符串算旅行天数
from datetime import datetime
# 导入 requests：直接 HTTP 调 Amadeus（不用官方 SDK）
import requests
# 从 dotenv 导入 load_dotenv：把 .env 读进环境变量
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：经 OpenRouter 调云端模型
from openai import OpenAI
# 从 IPython.display 导入 Markdown、display：在笔记本里渲染推荐结果
from IPython.display import Markdown, display

# Amadeus 测试环境：OAuth2 client_credentials 换 token 的地址
AMADEUS_AUTH_URL = "https://test.api.amadeus.com/v1/security/oauth2/token"
# Amadeus 测试环境：航班报价（flight-offers）查询地址
AMADEUS_FLIGHTS_URL = "https://test.api.amadeus.com/v2/shopping/flight-offers"


In [ ]:
# ========== 环境设置：读密钥并做一次「有没有配好」的打印检查 ==========

# 加载 .env；override=True 用文件覆盖已有同名环境变量
load_dotenv(override=True)
# OpenRouter 密钥：后面 OpenAI 客户端要用
openrouter_key = os.getenv("OPENROUTER_API_KEY")
# Amadeus 应用 ID / Secret：没有则后面走 mock 航班数据
amadeus_id = os.getenv("AMADEUS_CLIENT_ID")
amadeus_secret = os.getenv("AMADEUS_CLIENT_SECRET")

# 打印检查结果：文案 "OK" / "missing" / "mock" 保持英文（与原逻辑一致）
print("OpenRouter:", "OK" if openrouter_key else "missing")
print("Amadeus:", "OK" if (amadeus_id and amadeus_secret) else "mock")


In [ ]:
# ========== Amadeus API：OAuth 换 token + 查航班（纯 HTTP，无 SDK） ==========

# 用 client_credentials 向 Amadeus 换 access_token；失败返回 None
def get_amadeus_token():
    # POST 表单：grant_type + client_id + client_secret；Content-Type 为 urlencoded
    r = requests.post(AMADEUS_AUTH_URL, data={"grant_type": "client_credentials", "client_id": amadeus_id, "client_secret": amadeus_secret},
                      headers={"Content-Type": "application/x-www-form-urlencoded"})
    # 仅 200 时取 access_token；否则 None
    return r.json().get("access_token") if r.status_code == 200 else None

# 按起降机场、往返日期、人数查报价；最多返回前 7 条；失败返回 None
def amadeus_fetch_flights(origin, destination, departure_date, return_date, adults=1, children=0):
    # 先拿 Bearer token
    token = get_amadeus_token()
    # 拿不到 token 直接失败
    if not token:
        return None
    # 查询参数：IATA 码、日期、成人数（字段名必须按 Amadeus 文档）
    params = {"originLocationCode": origin, "destinationLocationCode": destination, "departureDate": departure_date,
              "returnDate": return_date, "adults": adults}
    # 有儿童才附加 children 参数
    if children:
        params["children"] = children
    # GET flight-offers，Authorization: Bearer <token>
    r = requests.get(AMADEUS_FLIGHTS_URL, params=params, headers={"Authorization": f"Bearer {token}"})
    # 成功则取 data 列表前 7 条；失败返回 None（空列表仍算成功，用 or []）
    return (r.json().get("data") or [])[:7] if r.status_code == 200 else None


In [ ]:
# ========== 冒烟测试：有凭证才真调 Amadeus，否则打印缺凭证 ==========

# 返回 True/False，并打印 offers 条数（便于确认网络与密钥）
def test_amadeus_api():
    # 缺任一凭证：打印英文提示并返回 False（文案保持原文）
    if not amadeus_id or not amadeus_secret:
        print("Missing Amadeus credentials")
        return False
    # 示例航线 JFK→LHR，日期与 adults 保持原文
    offers = amadeus_fetch_flights("JFK", "LHR", "2026-06-01", "2026-06-08", adults=1)
    # None 表示 HTTP/鉴权失败；空列表仍算 OK
    ok = offers is not None
    print(f"Amadeus: {'OK' if ok else 'error'} - {len(offers or [])} offers")
    return ok

# 运行单元格时立刻执行一次测试
test_amadeus_api()


In [ ]:
# ========== TicketRecommender：拉航班（或 mock）+ 让 LLM 出 Top 3 推荐 ==========

class TicketRecommender:
    # system prompt 保持英文：旅行专家角色与输出格式要求（改译会改行为）
    SYSTEM_PROMPT = """You are a travel expert. Recommend top 3 tickets ranked 1-3. 
    Consider price, airline rating, value. Use full location names, show duration, adults/children. 
    Include booking links as [Book here](url). Markdown. 
    Add some explanation for your reason for each recommendation"""

    # 默认模型 id 走 OpenRouter 风格命名 openai/gpt-4o-mini
    def __init__(self, model="openai/gpt-4o-mini"):
        # 保存模型名，create 时传入
        self.model = model
        # OpenRouter 的 OpenAI 兼容端点；密钥从环境变量读
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

    # 对外主入口：组装输入 → 取航班 → 调模型写推荐
    def recommend(self, origin, destination, start_date, return_date, adults=1, children=0, preference="value_for_money"):
        # 用返回日减出发日，得到旅行天数（duration_days）
        duration_days = (datetime.strptime(return_date, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")).days
        # 结构化用户行程，稍后 json.dumps 进 prompt
        input_data = {"origin": origin, "destination": destination, "start_date": start_date, "return_date": return_date,
                      "duration_days": duration_days, "adults": adults, "children": children}
        # 真 API 或 mock
        flight_data = self._fetch_flight_data(origin, destination, start_date, return_date, adults, children)
        # 若返回带 "error" 键的字典，直接把错误字符串交给调用方
        if isinstance(flight_data, dict) and "error" in flight_data:
            return flight_data["error"]
        # 正常路径：把行程 + 航班 + 偏好交给模型
        return self._get_ai_recommendation(input_data, flight_data, preference)

    # 有 Amadeus 凭证则真查；失败返回 error 字典；无凭证或空结果用 mock
    def _fetch_flight_data(self, origin, destination, start_date, return_date, adults=1, children=0):
        if amadeus_id and amadeus_secret:
            offers = amadeus_fetch_flights(origin, destination, start_date, return_date, adults, children)
            if offers is not None:
                # 非空用真数据；空列表退回 mock，避免模型无料可荐
                return offers if offers else self._mock_flight_data()
            # HTTP/鉴权失败：错误字符串保持英文
            return {"error": "Travel API error"}
        # 未配置凭证：直接 mock
        return self._mock_flight_data()

    # 本地假数据：三家航司示例（价格/评分/经停/预订链接），便于无 API 时演示
    def _mock_flight_data(self):
        return {"flights": [{"airline": "British Airways", "price": 450, "rating": 4.2, "stops": 0, "link": "https://www.britishairways.com/travel/book/public/en_us"},
                {"airline": "United Airlines", "price": 380, "rating": 3.8, "stops": 1, "link": "https://www.united.com/en/us/book-flight"},
                {"airline": "Delta", "price": 520, "rating": 4.5, "stops": 0, "link": "https://www.delta.com/flight-search"}]}

    # 把行程与航班 JSON 拼进 user 消息，调用 Chat Completions
    def _get_ai_recommendation(self, input_data, api_response, preference):
        # prompt 模板与 Preference 等关键词保持英文（影响模型的可运行字符串）
        prompt = f"User: {json.dumps(input_data)}\nFlights: {json.dumps(api_response)}\nPreference: {preference}\nTop 3 recommendations."
        try:
            # system + user 两条 messages；非流式拿完整 content
            r = self.client.chat.completions.create(model=self.model, messages=[{"role": "system", "content": self.SYSTEM_PROMPT}, {"role": "user", "content": prompt}])
            return r.choices[0].message.content
        except Exception as e:
            # 异常信息前缀保持英文 AI error:
            return f"AI error: {e}"


In [ ]:
# ========== 演示：拉斯维加斯 → 伦敦，按「性价比」偏好要 Top 推荐 ==========

# 实例化推荐器（默认 OpenRouter + openai/gpt-4o-mini）
recommender = TicketRecommender()
# 改 origin/destination/日期/人数/preference 即可换场景重跑
result = recommender.recommend(
    origin="LAS",
    destination="LHR",
    start_date="2026-04-01",
    return_date="2026-04-08",
    adults=2,
    children=1,
    preference="value_for_money"
)
# 模型返回 Markdown 字符串，用 display 漂亮渲染
display(Markdown(result))
